# Config Files - Answers

Python programs often need configuration: API keys, database URLs, feature flags, and other settings that vary between environments. Hard-coding these values is fragile and insecure.

Two common approaches for managing configuration in Python:

1. **Environment variables** -- ideal for secrets and deployment-specific settings
2. **Config files** -- useful for structured application settings with multiple sections

In this notebook, you will practice both approaches.

## 1. Environment Variables

Environment variables are key-value pairs maintained by your operating system. Every process inherits a copy of these variables when it starts.

In Python, `os.environ` is a dict-like object that gives you access to all environment variables of the current process. You can read, set, and delete them just like dictionary entries.

Cloud platforms (Azure Functions, AWS Lambda, Docker containers) use environment variables extensively to pass secrets and configuration to applications. This keeps sensitive values out of source code.

```python
import os

# Read an environment variable
value = os.environ['HOME']

# Read with a default (returns None if not found)
value = os.environ.get('MY_SETTING', 'default_value')

# Set an environment variable
os.environ['MY_SETTING'] = 'some_value'
```

### Exercise 1

Import the `os` module and print the value of the `PATH` environment variable.

**Explanation:** We access the `PATH` environment variable through `os.environ`, which behaves like a dictionary. Using dictionary key access (`os.environ['PATH']`) retrieves the value directly. This variable contains the list of directories the OS searches for executable programs, separated by colons on Unix/macOS or semicolons on Windows.

In [ ]:
import os

print(os.environ['PATH'])

### Exercise 2

Set a new environment variable `MY_API_KEY` to `"secret-key-123"` using `os.environ`, then read it back and print it.

**Explanation:** We assign a value to `os.environ['MY_API_KEY']` just like setting a dictionary entry. This creates the environment variable in the current process. Reading it back confirms the variable was set correctly. Note that environment variables set this way only persist for the lifetime of the current Python process.

In [ ]:
import os

os.environ['MY_API_KEY'] = 'secret-key-123'

print(os.environ['MY_API_KEY'])

### Exercise 3

Write a function `get_config(key, default=None)` that reads an environment variable, returning the default value if the variable doesn't exist. Test it with an existing key (e.g. `PATH`) and a non-existing key (e.g. `NON_EXISTENT_VAR`).

**Explanation:** We use `os.environ.get(key, default)` which is the dict-like `.get()` method. Unlike direct dictionary access (`os.environ[key]`), `.get()` does not raise a `KeyError` when the key is missing -- instead it returns the specified default value. This pattern is very common for optional configuration with sensible fallbacks.

In [ ]:
import os

def get_config(key, default=None):
    return os.environ.get(key, default)

# Test with an existing key
print(f"PATH: {get_config('PATH')[:80]}...")  # Truncated for readability

# Test with a non-existing key
print(f"NON_EXISTENT_VAR: {get_config('NON_EXISTENT_VAR', 'default_value')}")

## 2. Config Files with `configparser`

For structured application settings, Python's built-in `configparser` module reads and writes INI-style configuration files. These files are organized into **sections**, each containing **key-value pairs**.

An INI file looks like this:

```ini
[DEFAULT]
timeout = 30
retries = 3

[database]
host = localhost
port = 5432
name = mydb

[api]
base_url = https://api.example.com
```

The `[DEFAULT]` section is special: its values serve as fallback defaults for all other sections.

Basic usage:

```python
import configparser

config = configparser.ConfigParser()
config.read('config.ini')

# Read a value
value = config.get('database', 'host')

# Add a new section
config.add_section('new_section')
config.set('new_section', 'key', 'value')

# Write changes back to file
with open('config.ini', 'w') as f:
    config.write(f)
```

See the [configparser documentation](https://docs.python.org/3/library/configparser.html) for more details.

### Exercise 4

Read `config.ini` using `configparser.ConfigParser()`. Print the value of `port` from the `topsecret.server.example` section.

**Explanation:** We create a `ConfigParser` instance and call `.read()` with the filename. The `.get(section, key)` method retrieves the value as a string. Note that all values in configparser are stored as strings, so you may need to convert them (e.g. `int()`) if you need a different type.

In [ ]:
import configparser

config = configparser.ConfigParser()
config.read('config.ini')

port = config.get('topsecret.server.example', 'port')
print(f"Port: {port}")

### Exercise 5

Add a new section called `database` with the following keys:
- `host` = `localhost`
- `port` = `5432`
- `name` = `mydb`

Write the updated config back to `config.ini`.

**Explanation:** We first read the existing config, then use `.add_section()` to create a new section and `.set()` to add key-value pairs. Finally, we open the file in write mode and call `config.write()` to persist the changes. A common pitfall is forgetting to read the file first, which would overwrite the existing content with only the new section.

In [ ]:
import configparser

config = configparser.ConfigParser()
config.read('config.ini')

config.add_section('database')
config.set('database', 'host', 'localhost')
config.set('database', 'port', '5432')
config.set('database', 'name', 'mydb')

with open('config.ini', 'w') as configfile:
    config.write(configfile)

print("Config updated successfully.")

### Exercise 6

Write a function `read_config(filepath, section, key)` that reads a specific value from a config file. If the section or key doesn't exist, return `None`. Test it with a valid section/key and an invalid one.

**Explanation:** We use `config.has_section()` and `config.has_option()` to check whether the section and key exist before reading. This avoids `NoSectionError` and `NoOptionError` exceptions. An alternative approach would be to wrap the `.get()` call in a try/except block, but explicit checks make the intent clearer.

In [ ]:
import configparser

def read_config(filepath, section, key):
    config = configparser.ConfigParser()
    config.read(filepath)
    
    if not config.has_section(section):
        return None
    if not config.has_option(section, key):
        return None
    
    return config.get(section, key)

# Test with a valid section and key
print(read_config('config.ini', 'topsecret.server.example', 'port'))

# Test with an invalid section
print(read_config('config.ini', 'nonexistent', 'key'))

# Test with a valid section but invalid key
print(read_config('config.ini', 'forge.example', 'nonexistent_key'))

### Exercise 7

Create a small script that reads database connection settings (`host`, `port`, `name`) from the `database` section of `config.ini` and prints a connection string in the format `host:port/name`.

**Note:** If you haven't completed Exercise 5, the `database` section may not exist in `config.ini` yet. Complete Exercise 5 first, or manually add the section.

**Explanation:** We read the three database settings from the `database` section and combine them into a connection string using an f-string. This pattern is common when constructing database URIs (e.g. for SQLAlchemy). In production code, you would also handle missing values and add the protocol prefix (e.g. `postgresql://`).

In [ ]:
import configparser

config = configparser.ConfigParser()
config.read('config.ini')

host = config.get('database', 'host')
port = config.get('database', 'port')
name = config.get('database', 'name')

connection_string = f"{host}:{port}/{name}"
print(f"Connection string: {connection_string}")

## Summary

In this notebook you practiced two common approaches for configuration in Python:

- **Environment variables** (`os.environ`) -- best for secrets and values that change per deployment environment. Cloud platforms like Azure Functions and Docker rely heavily on this mechanism.
- **Config files** (`configparser`) -- best for structured settings with multiple sections. INI files are human-readable and easy to version-control.

In practice, many applications combine both: config files for application defaults, and environment variables for secrets and environment-specific overrides.